In [1]:
from pathlib import Path
import pandas as pd

print("Pandas版本：", pd.__version__)
print("当前工作目录：", Path.cwd())

Pandas版本： 3.0.3
当前工作目录： /Users/nyx/Desktop/user-behavior-analysis/notebooks


In [2]:
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

print("数据文件是否存在：", DATA_PATH.exists())
print("数据文件完整路径：", DATA_PATH.resolve())
print("文件大小：", round(DATA_PATH.stat().st_size / 1024**2, 2), "MB")

数据文件是否存在： True
数据文件完整路径： /Users/nyx/Desktop/user-behavior-analysis/data/raw/user_behavior_processed.csv
文件大小： 469.46 MB


In [3]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=10000
)

print("样本数据形状：", sample_df.shape)
display(sample_df.head())

样本数据形状： (10000, 5)


,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02,98047837,232431562,4245,1
1,2025-12-09 20,97726136,383583590,5894,1
2,2025-12-18 11,98607707,64749712,2883,1
3,2025-12-06 10,98662432,320593836,6562,1
4,2025-12-16 21,98145908,290208520,13926,1


In [4]:
print("字段名称：")
print(sample_df.columns.tolist())

print("\n各字段的数据类型：")
print(sample_df.dtypes)

print("\n数据基本信息：")
sample_df.info()

字段名称：
['time', 'user_id', 'item_id', 'item_category', 'behavior_type']

各字段的数据类型：
time               str
user_id          int64
item_id          int64
item_category    int64
behavior_type    int64
dtype: object

数据基本信息：
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time           10000 non-null  str  
 1   user_id        10000 non-null  int64
 2   item_id        10000 non-null  int64
 3   item_category  10000 non-null  int64
 4   behavior_type  10000 non-null  int64
dtypes: int64(4), str(1)
memory usage: 517.7 KB


In [5]:
print("各字段缺失值数量：")
print(sample_df.isna().sum())

print("\n完全重复行数量：")
print(sample_df.duplicated().sum())

print("\n行为类型分布：")
print(sample_df["behavior_type"].value_counts().sort_index())

各字段缺失值数量：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

完全重复行数量：
717

行为类型分布：
behavior_type
1    9430
2     195
3     276
4      99
Name: count, dtype: int64


In [6]:
sample_df["time"] = pd.to_datetime(
    sample_df["time"],
    format="%Y-%m-%d %H",
    errors="coerce"
)

print("转换后的字段类型：")
print(sample_df.dtypes)

print("\n无法转换的时间数量：")
print(sample_df["time"].isna().sum())

print("\n最早时间：", sample_df["time"].min())
print("最晚时间：", sample_df["time"].max())

转换后的字段类型：
time             datetime64[us]
user_id                   int64
item_id                   int64
item_category             int64
behavior_type             int64
dtype: object

无法转换的时间数量：
0

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [7]:
duplicate_rows = sample_df[
    sample_df.duplicated(keep=False)
].sort_values(
    ["user_id", "item_id", "time", "behavior_type"]
)

print("重复记录数量：", len(duplicate_rows))
display(duplicate_rows.head(20))

重复记录数量： 1395


,time,user_id,item_id,item_category,behavior_type
9845,2025-12-03 11:00:00,2967122,17123536,11178,1
9854,2025-12-03 11:00:00,2967122,17123536,11178,1
9467,2025-12-03 16:00:00,2967122,176322914,2866,1
9503,2025-12-03 16:00:00,2967122,176322914,2866,1
9431,2025-11-22 05:00:00,2967122,272639944,5027,1
9476,2025-11-22 05:00:00,2967122,272639944,5027,1
9611,2025-11-21 15:00:00,2967122,273419428,169,1
9647,2025-11-21 15:00:00,2967122,273419428,169,1
2530,2025-11-29 21:00:00,10131505,66223566,4677,1
2908,2025-11-29 21:00:00,10131505,66223566,4677,1


In [8]:
behavior_mapping = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_summary = (
    sample_df["behavior_type"]
    .value_counts()
    .sort_index()
    .rename_axis("behavior_type")
    .reset_index(name="count")
)

behavior_summary["behavior_name"] = behavior_summary["behavior_type"].map(
    behavior_mapping
)

behavior_summary["percentage"] = (
    behavior_summary["count"] / len(sample_df) * 100
).round(2)

display(behavior_summary)

,behavior_type,count,behavior_name,percentage
0,1,9430,浏览,94.30
1,2,195,收藏,1.95
2,3,276,加购,2.76
3,4,99,购买,0.99


In [9]:
dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

chunk_size = 500_000

chunk_reader = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
)

print("分块读取器创建成功")



分块读取器创建成功


In [10]:
total_rows = 0
missing_counts = None
behavior_counts = {}
min_time = None
max_time = None
chunk_count = 0

for chunk in pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
):
    chunk_count += 1
    total_rows += len(chunk)

    # 缺失值统计
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # 行为类型统计
    current_behavior = chunk["behavior_type"].value_counts()

    for behavior_type, count in current_behavior.items():
        behavior_counts[behavior_type] = (
            behavior_counts.get(behavior_type, 0) + count
        )

    # 时间范围
    current_min_time = chunk["time"].min()
    current_max_time = chunk["time"].max()

    if min_time is None or current_min_time < min_time:
        min_time = current_min_time

    if max_time is None or current_max_time > max_time:
        max_time = current_max_time

    print(f"已完成第 {chunk_count} 个数据块，累计 {total_rows:,} 行")

    

已完成第 1 个数据块，累计 500,000 行
已完成第 2 个数据块，累计 1,000,000 行
已完成第 3 个数据块，累计 1,500,000 行
已完成第 4 个数据块，累计 2,000,000 行
已完成第 5 个数据块，累计 2,500,000 行
已完成第 6 个数据块，累计 3,000,000 行
已完成第 7 个数据块，累计 3,500,000 行
已完成第 8 个数据块，累计 4,000,000 行
已完成第 9 个数据块，累计 4,500,000 行
已完成第 10 个数据块，累计 5,000,000 行
已完成第 11 个数据块，累计 5,500,000 行
已完成第 12 个数据块，累计 6,000,000 行
已完成第 13 个数据块，累计 6,500,000 行
已完成第 14 个数据块，累计 7,000,000 行
已完成第 15 个数据块，累计 7,500,000 行
已完成第 16 个数据块，累计 8,000,000 行
已完成第 17 个数据块，累计 8,500,000 行
已完成第 18 个数据块，累计 9,000,000 行
已完成第 19 个数据块，累计 9,500,000 行
已完成第 20 个数据块，累计 10,000,000 行
已完成第 21 个数据块，累计 10,500,000 行
已完成第 22 个数据块，累计 11,000,000 行
已完成第 23 个数据块，累计 11,500,000 行
已完成第 24 个数据块，累计 12,000,000 行
已完成第 25 个数据块，累计 12,256,906 行


In [11]:
print("完整数据总行数：", f"{total_rows:,}")
print("数据块数量：", chunk_count)

print("\n各字段缺失值：")
print(missing_counts.astype("int64"))

print("\n行为类型数量：")
for behavior_type in sorted(behavior_counts):
    print(
        behavior_type,
        behavior_mapping[behavior_type],
        f"{behavior_counts[behavior_type]:,}"
    )

print("\n最早时间：", min_time)
print("最晚时间：", max_time)

完整数据总行数： 12,256,906
数据块数量： 25

各字段缺失值：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

行为类型数量：
1 浏览 11,550,581
2 收藏 242,556
3 加购 343,564
4 购买 120,205

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [12]:
full_behavior_summary = pd.DataFrame(
    {
        "behavior_type": list(behavior_counts.keys()),
        "count": list(behavior_counts.values())
    }
).sort_values("behavior_type")

full_behavior_summary["behavior_name"] = (
    full_behavior_summary["behavior_type"]
    .map(behavior_mapping)
)

full_behavior_summary["percentage"] = (
    full_behavior_summary["count"]
    / total_rows
    * 100
).round(2)

display(full_behavior_summary)


,behavior_type,count,behavior_name,percentage
0,1,11550581,浏览,94.24
2,2,242556,收藏,1.98
1,3,343564,加购,2.80
3,4,120205,购买,0.98


In [13]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    memory_map=True
)

print("全部数据读取完成")
print("数据行列数：", df.shape)
print("总行数：", f"{len(df):,}")
print("总列数：", df.shape[1])
print(
    "内存占用：",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)



全部数据读取完成
数据行列数： (12256906, 5)
总行数： 12,256,906
总列数： 5
内存占用： 338.98 MB


In [14]:
print(df.shape)

(12256906, 5)


In [15]:
from pathlib import Path
from collections import Counter
import pandas as pd

# Notebook 位于 notebooks 文件夹时使用这个路径
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")
CHUNK_SIZE = 500_000

total_rows = 0
user_ids = set()
item_ids = set()
category_ids = set()

behavior_counts = Counter()
behavior_users = {1: set(), 2: set(), 3: set(), 4: set()}

min_time = None
max_time = None
missing_counts = Counter()

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=CHUNK_SIZE
):
    total_rows += len(chunk)

    # 去重主体数量
    user_ids.update(chunk["user_id"].dropna().unique())
    item_ids.update(chunk["item_id"].dropna().unique())
    category_ids.update(chunk["item_category"].dropna().unique())

    # 缺失值统计
    missing_counts.update(chunk.isna().sum().to_dict())

    # 四种行为统计
    behavior = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    )

    valid_behavior = behavior.dropna().astype(int)
    behavior_counts.update(valid_behavior.tolist())

    for behavior_code in [1, 2, 3, 4]:
        users = chunk.loc[
            behavior == behavior_code, "user_id"
        ].dropna().unique()

        behavior_users[behavior_code].update(users)

    # 时间范围
    parsed_time = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    ).dropna()

    if not parsed_time.empty:
        chunk_min = parsed_time.min()
        chunk_max = parsed_time.max()

        min_time = (
            chunk_min
            if min_time is None
            else min(min_time, chunk_min)
        )

        max_time = (
            chunk_max
            if max_time is None
            else max(max_time, chunk_max)
        )

# 数据概况表
overview = pd.DataFrame({
    "指标": [
        "行为记录总数",
        "去重用户数",
        "去重商品数",
        "商品品类数",
        "开始时间",
        "结束时间"
    ],
    "结果": [
        f"{total_rows:,}",
        f"{len(user_ids):,}",
        f"{len(item_ids):,}",
        f"{len(category_ids):,}",
        min_time,
        max_time
    ]
})

behavior_names = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_table = pd.DataFrame([
    {
        "行为类型": behavior_names[code],
        "行为次数": behavior_counts[code],
        "行为占比": (
            behavior_counts[code] / total_rows
            if total_rows else 0
        ),
        "涉及用户数": len(behavior_users[code])
    }
    for code in [1, 2, 3, 4]
])

behavior_table["行为次数"] = behavior_table["行为次数"].map(
    lambda x: f"{x:,}"
)
behavior_table["涉及用户数"] = behavior_table["涉及用户数"].map(
    lambda x: f"{x:,}"
)
behavior_table["行为占比"] = behavior_table["行为占比"].map(
    lambda x: f"{x:.2%}"
)

print("一、数据基础概况")
display(overview)

print("\n二、用户行为分布")
display(behavior_table)

print("\n三、各字段缺失值")
display(pd.DataFrame({
    "字段": columns,
    "缺失数量": [missing_counts[col] for col in columns]
}))


一、数据基础概况


,指标,结果
0,行为记录总数,"12,256,906"
1,去重用户数,"10,000"
2,去重商品数,"2,876,947"
3,商品品类数,"8,916"
4,开始时间,2025-11-18 00:00:00
5,结束时间,2025-12-18 23:00:00



二、用户行为分布


,行为类型,行为次数,行为占比,涉及用户数
0,浏览,"11,550,581",94.24%,"10,000"
1,收藏,"242,556",1.98%,"6,730"
2,加购,"343,564",2.80%,"8,614"
3,购买,"120,205",0.98%,"8,886"



三、各字段缺失值


,字段,缺失数量
0,time,0
1,user_id,0
2,item_id,0
3,item_category,0
4,behavior_type,0


In [16]:
from pathlib import Path
from collections import Counter
import pandas as pd

# 自动寻找原始数据
possible_paths = [
    Path("../data/raw/user_behavior_processed.csv"),
    Path("data/raw/user_behavior_processed.csv")
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError("没有找到 user_behavior_processed.csv")

print("读取文件：", DATA_PATH)

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

total_rows = 0
users = set()
items = set()
categories = set()
behavior_counts = Counter()

start_time = None
end_time = None

# 分块读取，不会一次性占满内存
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=500_000
):
    total_rows += len(chunk)

    # 这里只是统计唯一数量，不是删除重复行为
    users.update(chunk["user_id"].dropna().unique())
    items.update(chunk["item_id"].dropna().unique())
    categories.update(chunk["item_category"].dropna().unique())

    behavior = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    ).dropna().astype(int)

    behavior_counts.update(behavior.tolist())

    times = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    ).dropna()

    if not times.empty:
        chunk_start = times.min()
        chunk_end = times.max()

        start_time = (
            chunk_start
            if start_time is None
            else min(start_time, chunk_start)
        )

        end_time = (
            chunk_end
            if end_time is None
            else max(end_time, chunk_end)
        )

print("\n===== 原始数据基础统计 =====")
print(f"行为记录总数：{total_rows:,}")
print(f"用户数量：{len(users):,}")
print(f"商品数量：{len(items):,}")
print(f"商品品类数量：{len(categories):,}")
print(f"时间范围：{start_time} 至 {end_time}")

print("\n===== 行为数量 =====")
print(f"浏览：{behavior_counts[1]:,}")
print(f"收藏：{behavior_counts[2]:,}")
print(f"加购：{behavior_counts[3]:,}")
print(f"购买：{behavior_counts[4]:,}")

读取文件： ../data/raw/user_behavior_processed.csv

===== 原始数据基础统计 =====
行为记录总数：12,256,906
用户数量：10,000
商品数量：2,876,947
商品品类数量：8,916
时间范围：2025-11-18 00:00:00 至 2025-12-18 23:00:00

===== 行为数量 =====
浏览：11,550,581
收藏：242,556
加购：343,564
购买：120,205


In [17]:
import pandas as pd

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

missing_counts = pd.Series(
    0,
    index=columns,
    dtype="int64"
)

invalid_id_counts = {
    "user_id": 0,
    "item_id": 0,
    "item_category": 0
}

invalid_behavior_count = 0
invalid_time_count = 0
problem_row_count = 0
checked_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=columns,
        chunksize=500_000
    ),
    start=1
):
    checked_rows += len(chunk)

    # 1. 检查缺失值
    missing_counts += chunk.isna().sum()

    # 标记本批数据中存在问题的行
    problem_mask = chunk.isna().any(axis=1)

    # 2. 检查用户、商品和品类ID
    for column in [
        "user_id",
        "item_id",
        "item_category"
    ]:
        numeric_values = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

        invalid_id_mask = (
            chunk[column].notna()
            & (
                numeric_values.isna()
                | (numeric_values <= 0)
            )
        )

        invalid_id_counts[column] += int(
            invalid_id_mask.sum()
        )

        problem_mask |= invalid_id_mask

    # 3. 检查行为编码是否属于1、2、3、4
    behavior_values = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    )

    invalid_behavior_mask = (
        chunk["behavior_type"].notna()
        & ~behavior_values.isin([1, 2, 3, 4])
    )

    invalid_behavior_count += int(
        invalid_behavior_mask.sum()
    )

    problem_mask |= invalid_behavior_mask

    # 4. 检查时间能否正常解析
    parsed_time = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    )

    invalid_time_mask = (
        chunk["time"].notna()
        & parsed_time.isna()
    )

    invalid_time_count += int(
        invalid_time_mask.sum()
    )

    problem_mask |= invalid_time_mask

    # 存在至少一个问题的行数
    problem_row_count += int(problem_mask.sum())

    print(
        f"已检查第 {chunk_number} 个数据块，"
        f"累计 {checked_rows:,} 行"
    )

print("\n===== 数据质量检查结果 =====")

quality_table = pd.DataFrame({
    "检查项目": [
        "总记录数",
        "存在至少一个问题的记录数",
        "user_id非法值",
        "item_id非法值",
        "item_category非法值",
        "behavior_type非法值",
        "time无法解析"
    ],
    "数量": [
        checked_rows,
        problem_row_count,
        invalid_id_counts["user_id"],
        invalid_id_counts["item_id"],
        invalid_id_counts["item_category"],
        invalid_behavior_count,
        invalid_time_count
    ]
})

display(quality_table)

print("\n===== 各字段缺失值 =====")

missing_table = pd.DataFrame({
    "字段": columns,
    "缺失数量": [
        int(missing_counts[column])
        for column in columns
    ],
    "缺失率": [
        f"{missing_counts[column] / checked_rows:.4%}"
        for column in columns
    ]
})

display(missing_table)

已检查第 1 个数据块，累计 500,000 行
已检查第 2 个数据块，累计 1,000,000 行
已检查第 3 个数据块，累计 1,500,000 行
已检查第 4 个数据块，累计 2,000,000 行
已检查第 5 个数据块，累计 2,500,000 行
已检查第 6 个数据块，累计 3,000,000 行
已检查第 7 个数据块，累计 3,500,000 行
已检查第 8 个数据块，累计 4,000,000 行
已检查第 9 个数据块，累计 4,500,000 行
已检查第 10 个数据块，累计 5,000,000 行
已检查第 11 个数据块，累计 5,500,000 行
已检查第 12 个数据块，累计 6,000,000 行
已检查第 13 个数据块，累计 6,500,000 行
已检查第 14 个数据块，累计 7,000,000 行
已检查第 15 个数据块，累计 7,500,000 行
已检查第 16 个数据块，累计 8,000,000 行
已检查第 17 个数据块，累计 8,500,000 行
已检查第 18 个数据块，累计 9,000,000 行
已检查第 19 个数据块，累计 9,500,000 行
已检查第 20 个数据块，累计 10,000,000 行
已检查第 21 个数据块，累计 10,500,000 行
已检查第 22 个数据块，累计 11,000,000 行
已检查第 23 个数据块，累计 11,500,000 行
已检查第 24 个数据块，累计 12,000,000 行
已检查第 25 个数据块，累计 12,256,906 行

===== 数据质量检查结果 =====


,检查项目,数量
0,总记录数,12256906
1,存在至少一个问题的记录数,0
2,user_id非法值,0
3,item_id非法值,0
4,item_category非法值,0
5,behavior_type非法值,0
6,time无法解析,0



===== 各字段缺失值 =====


,字段,缺失数量,缺失率
0,time,0,0.0000%
1,user_id,0,0.0000%
2,item_id,0,0.0000%
3,item_category,0,0.0000%
4,behavior_type,0,0.0000%


In [18]:
import pandas as pd

numeric_columns = [
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

value_ranges = {
    column: {
        "最小值": None,
        "最大值": None
    }
    for column in numeric_columns
}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=numeric_columns,
        chunksize=500_000
    ),
    start=1
):
    for column in numeric_columns:
        values = pd.to_numeric(
            chunk[column],
            errors="raise"
        )

        current_min = int(values.min())
        current_max = int(values.max())

        old_min = value_ranges[column]["最小值"]
        old_max = value_ranges[column]["最大值"]

        value_ranges[column]["最小值"] = (
            current_min
            if old_min is None
            else min(old_min, current_min)
        )

        value_ranges[column]["最大值"] = (
            current_max
            if old_max is None
            else max(old_max, current_max)
        )

    print(f"已检查第 {chunk_number} 个数据块")

def recommend_dtype(min_value, max_value):
    if min_value >= 0:
        if max_value <= 255:
            return "uint8"
        elif max_value <= 65_535:
            return "uint16"
        elif max_value <= 4_294_967_295:
            return "uint32"
        else:
            return "uint64"
    else:
        if -128 <= min_value and max_value <= 127:
            return "int8"
        elif -32_768 <= min_value and max_value <= 32_767:
            return "int16"
        elif (
            -2_147_483_648 <= min_value
            and max_value <= 2_147_483_647
        ):
            return "int32"
        else:
            return "int64"

range_table = pd.DataFrame([
    {
        "字段": column,
        "最小值": value_ranges[column]["最小值"],
        "最大值": value_ranges[column]["最大值"],
        "建议类型": recommend_dtype(
            value_ranges[column]["最小值"],
            value_ranges[column]["最大值"]
        )
    }
    for column in numeric_columns
])

print("\n===== 字段范围与建议类型 =====")
display(range_table)


已检查第 1 个数据块
已检查第 2 个数据块
已检查第 3 个数据块
已检查第 4 个数据块
已检查第 5 个数据块
已检查第 6 个数据块
已检查第 7 个数据块
已检查第 8 个数据块
已检查第 9 个数据块
已检查第 10 个数据块
已检查第 11 个数据块
已检查第 12 个数据块
已检查第 13 个数据块
已检查第 14 个数据块
已检查第 15 个数据块
已检查第 16 个数据块
已检查第 17 个数据块
已检查第 18 个数据块
已检查第 19 个数据块
已检查第 20 个数据块
已检查第 21 个数据块
已检查第 22 个数据块
已检查第 23 个数据块
已检查第 24 个数据块
已检查第 25 个数据块

===== 字段范围与建议类型 =====


,字段,最小值,最大值,建议类型
0,user_id,4913,142455899,uint32
1,item_id,64,404562461,uint32
2,item_category,2,14080,uint16
3,behavior_type,1,4,uint8


In [19]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# 避免覆盖之前生成的文件
version = 1
OUTPUT_PATH = (
    processed_dir
    / f"user_behavior_standardized_v{version}.parquet"
)

while OUTPUT_PATH.exists():
    version += 1
    OUTPUT_PATH = (
        processed_dir
        / f"user_behavior_standardized_v{version}.parquet"
    )

writer = None
written_rows = 0

try:
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            DATA_PATH,
            usecols=columns,
            chunksize=500_000
        ),
        start=1
    ):
        # 统一时间格式
        chunk["time"] = pd.to_datetime(
            chunk["time"],
            errors="raise"
        )

        # 压缩字段类型
        chunk["user_id"] = chunk["user_id"].astype(
            "uint32"
        )
        chunk["item_id"] = chunk["item_id"].astype(
            "uint32"
        )
        chunk["item_category"] = chunk[
            "item_category"
        ].astype("uint16")
        chunk["behavior_type"] = chunk[
            "behavior_type"
        ].astype("uint8")

        # 固定字段顺序
        chunk = chunk[columns]

        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_PATH,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)
        written_rows += len(chunk)

        print(
            f"已写入第 {chunk_number} 个数据块，"
            f"累计 {written_rows:,} 行"
        )

finally:
    if writer is not None:
        writer.close()

# 验证生成结果
parquet_file = pq.ParquetFile(OUTPUT_PATH)
verified_rows = parquet_file.metadata.num_rows

csv_size_mb = DATA_PATH.stat().st_size / 1024**2
parquet_size_mb = OUTPUT_PATH.stat().st_size / 1024**2

print("\n===== 标准化数据生成结果 =====")
print("输出文件：", OUTPUT_PATH)
print(f"写入记录数：{written_rows:,}")
print(f"Parquet验证行数：{verified_rows:,}")
print(f"原始CSV大小：{csv_size_mb:.2f} MB")
print(f"Parquet大小：{parquet_size_mb:.2f} MB")
print(f"存储空间减少：{1 - parquet_size_mb / csv_size_mb:.2%}")

print("\n===== Parquet字段结构 =====")
print(parquet_file.schema_arrow)

已写入第 1 个数据块，累计 500,000 行
已写入第 2 个数据块，累计 1,000,000 行
已写入第 3 个数据块，累计 1,500,000 行
已写入第 4 个数据块，累计 2,000,000 行
已写入第 5 个数据块，累计 2,500,000 行
已写入第 6 个数据块，累计 3,000,000 行
已写入第 7 个数据块，累计 3,500,000 行
已写入第 8 个数据块，累计 4,000,000 行
已写入第 9 个数据块，累计 4,500,000 行
已写入第 10 个数据块，累计 5,000,000 行
已写入第 11 个数据块，累计 5,500,000 行
已写入第 12 个数据块，累计 6,000,000 行
已写入第 13 个数据块，累计 6,500,000 行
已写入第 14 个数据块，累计 7,000,000 行
已写入第 15 个数据块，累计 7,500,000 行
已写入第 16 个数据块，累计 8,000,000 行
已写入第 17 个数据块，累计 8,500,000 行
已写入第 18 个数据块，累计 9,000,000 行
已写入第 19 个数据块，累计 9,500,000 行
已写入第 20 个数据块，累计 10,000,000 行
已写入第 21 个数据块，累计 10,500,000 行
已写入第 22 个数据块，累计 11,000,000 行
已写入第 23 个数据块，累计 11,500,000 行
已写入第 24 个数据块，累计 12,000,000 行
已写入第 25 个数据块，累计 12,256,906 行

===== 标准化数据生成结果 =====
输出文件： ../data/processed/user_behavior_standardized_v1.parquet
写入记录数：12,256,906
Parquet验证行数：12,256,906
原始CSV大小：469.46 MB
Parquet大小：94.38 MB
存储空间减少：79.90%

===== Parquet字段结构 =====
time: timestamp[us]
user_id: uint32
item_id: uint32
item_category: uint16
behavior_type: uint8
-- schema

In [20]:
from pathlib import Path

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

REPORT_PATH = reports_dir / "data_quality_report.md"

behavior_names = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_lines = []

for code in [1, 2, 3, 4]:
    count = behavior_counts[code]
    ratio = count / total_rows

    behavior_lines.append(
        f"| {code} | {behavior_names[code]} | "
        f"{count:,} | {ratio:.2%} |"
    )

behavior_table_text = "\n".join(behavior_lines)

report_text = f"""# 淘宝用户行为数据质量校验报告

## 1. 数据基础概况

| 指标 | 统计结果 |
|---|---:|
| 行为记录总数 | {total_rows:,} |
| 用户数量 | {len(users):,} |
| 商品数量 | {len(items):,} |
| 商品品类数量 | {len(categories):,} |
| 开始时间 | {start_time} |
| 结束时间 | {end_time} |

## 2. 用户行为分布

| 行为编码 | 行为类型 | 行为次数 | 行为占比 |
|---:|---|---:|---:|
{behavior_table_text}

说明：行为占比反映各类行为记录在全部事件中的比例，
不直接等同于用户转化率。

## 3. 数据完整性检查

| 检查项目 | 结果 |
|---|---:|
| 存在至少一个问题的记录数 | {problem_row_count:,} |
| time缺失值 | {int(missing_counts["time"]):,} |
| user_id缺失值 | {int(missing_counts["user_id"]):,} |
| item_id缺失值 | {int(missing_counts["item_id"]):,} |
| item_category缺失值 | {int(missing_counts["item_category"]):,} |
| behavior_type缺失值 | {int(missing_counts["behavior_type"]):,} |
| user_id非法值 | {invalid_id_counts["user_id"]:,} |
| item_id非法值 | {invalid_id_counts["item_id"]:,} |
| item_category非法值 | {invalid_id_counts["item_category"]:,} |
| behavior_type非法值 | {invalid_behavior_count:,} |
| 无法解析的时间记录 | {invalid_time_count:,} |

## 4. 字段标准化结果

| 字段 | 标准化类型 |
|---|---|
| time | timestamp |
| user_id | uint32 |
| item_id | uint32 |
| item_category | uint16 |
| behavior_type | uint8 |

## 5. 重复行为处理原则

本阶段未直接删除任何重复行为记录。

同一用户对同一商品产生的重复浏览、收藏、加购和购买行为
具有实际业务意义，可以反映用户兴趣强度、购买意愿、
决策犹豫程度和复购行为。

后续特征工程将基于这些记录构建：

- 重复浏览次数；
- 重复收藏次数；
- 重复加购次数；
- 重复购买次数；
- 行为频率；
- 行为时间间隔。

## 6. 存储优化结果

| 指标 | 结果 |
|---|---:|
| 原始CSV大小 | {csv_size_mb:.2f} MB |
| Parquet文件大小 | {parquet_size_mb:.2f} MB |
| 存储空间减少 | {1 - parquet_size_mb / csv_size_mb:.2%} |
| Parquet记录数 | {verified_rows:,} |

## 7. 校验结论

原始数据共包含{total_rows:,}条用户行为记录。
所有核心字段均不存在缺失值或非法值，时间字段能够正常解析，
四类行为记录之和与数据总行数一致。

标准化处理完整保留了原始行为记录，并通过字段类型压缩和
Parquet列式存储将文件体积减少
{1 - parquet_size_mb / csv_size_mb:.2%}。

注意：数据文件的时间范围显示为2025年11月18日至12月18日，
最终使用前需要根据数据来源说明确认日期是否经过匿名化或平移。
"""

REPORT_PATH.write_text(
    report_text,
    encoding="utf-8"
)

print("数据质量报告已生成：")
print(REPORT_PATH)
print("\n===== 报告预览 =====\n")
print(report_text)

数据质量报告已生成：
../reports/data_quality_report.md

===== 报告预览 =====

# 淘宝用户行为数据质量校验报告

## 1. 数据基础概况

| 指标 | 统计结果 |
|---|---:|
| 行为记录总数 | 12,256,906 |
| 用户数量 | 10,000 |
| 商品数量 | 2,876,947 |
| 商品品类数量 | 8,916 |
| 开始时间 | 2025-11-18 00:00:00 |
| 结束时间 | 2025-12-18 23:00:00 |

## 2. 用户行为分布

| 行为编码 | 行为类型 | 行为次数 | 行为占比 |
|---:|---|---:|---:|
| 1 | 浏览 | 11,550,581 | 94.24% |
| 2 | 收藏 | 242,556 | 1.98% |
| 3 | 加购 | 343,564 | 2.80% |
| 4 | 购买 | 120,205 | 0.98% |

说明：行为占比反映各类行为记录在全部事件中的比例，
不直接等同于用户转化率。

## 3. 数据完整性检查

| 检查项目 | 结果 |
|---|---:|
| 存在至少一个问题的记录数 | 0 |
| time缺失值 | 0 |
| user_id缺失值 | 0 |
| item_id缺失值 | 0 |
| item_category缺失值 | 0 |
| behavior_type缺失值 | 0 |
| user_id非法值 | 0 |
| item_id非法值 | 0 |
| item_category非法值 | 0 |
| behavior_type非法值 | 0 |
| 无法解析的时间记录 | 0 |

## 4. 字段标准化结果

| 字段 | 标准化类型 |
|---|---|
| time | timestamp |
| user_id | uint32 |
| item_id | uint32 |
| item_category | uint16 |
| behavior_type | uint8 |

## 5. 重复行为处理原则

本阶段未直接删除任何重复行为记录。

同一用户对同一商品产生的重复浏览、收藏、加购和购买行为
具有实际业务意义，可以反映用户兴

In [21]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("..").resolve()
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

print("===== 当前 .gitignore =====")

if GITIGNORE_PATH.exists():
    print(GITIGNORE_PATH.read_text(encoding="utf-8"))
else:
    print("没有找到 .gitignore")

print("\n===== 当前 Git 状态 =====")

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout or "当前没有未提交的改动")

===== 当前 .gitignore =====
.venv/
__pycache__/
*.pyc
.DS_Store

.ipynb_checkpoints/
**/.ipynb_checkpoints/

data/raw/*
!data/raw/.gitkeep

data/processed/*
!data/processed/.gitkeep

outputs/*
!outputs/.gitkeep

*.db


===== 当前 Git 状态 =====
 M notebooks/01_data_loading.ipynb
?? reports/data_quality_report.md

